# DHL Logistics RAG System
## Retrieval-Augmented Generation with Official DHL Documentation

**Project Highlights:**
- Built using official DHL documents (Express Terms, Customs Guide, eCommerce Terms)
- Complete RAG pipeline: Load → Split → Embed → Store → Retrieve → Generate
- LangGraph ReAct Agent for multi-tool orchestration
- Local deployment with zero API cost (Ollama + ChromaDB)
- Comprehensive evaluation framework with test dataset

**Tech Stack:** Python, LangChain, LangGraph, ChromaDB, Ollama, Pandas

---
## Phase 1: Offline Indexing Pipeline
### Load → Split → Embed → Store

In [ ]:
# Step 1: Install Dependencies
import subprocess
import sys

packages = [
    "langchain",
    "langchain-community",
    "langchain-experimental",  # For SemanticChunker
    "langgraph",
    "langchain-ollama",
    "langchain-chroma",
    "langchain-text-splitters",
    "pypdf",
    "pandas",
    "matplotlib",
]

print("Installing dependencies...")
for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("✅ All dependencies installed successfully!")

In [ ]:
# Step 2: Verify Ollama is Running
import requests

OLLAMA_BASE_URL = "http://localhost:11434"

print("🔍 Checking Ollama server...")
try:
    response = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=5)
    if response.status_code == 200:
        models = response.json()
        print(f"✅ Ollama is running!")
        print(f"   Available models: {[m['name'] for m in models['models']]}")
    else:
        print("❌ Ollama server not responding")
except requests.exceptions.ConnectionError:
    print("❌ Ollama is not running")
    print("\n📌 Setup Instructions:")
    print("   1. Download from: https://ollama.ai/download")
    print("   2. Run in terminal: ollama pull mistral")

---
### Step 3: Load Documents

The first step in RAG is loading source documents from PDF files.

```
PDF Files → PyPDFLoader → List of Document Objects
```

In [ ]:
# Step 3: LOAD - Load DHL Official PDF Documents
from langchain_community.document_loaders import PyPDFLoader
import os

# PDF file paths
pdf_files = {
    "dhl_express_terms": "data/dhl_express_terms.pdf",      # DHL Express Terms & Conditions
    "dhl_customs_guide": "data/dhl_customs_guide.pdf",      # DHL US Customs Import Guide
    "dhl_ecommerce_terms": "data/dhl_ecommerce_terms.pdf",  # DHL eCommerce Terms
}

# Load all PDFs
all_documents = []

print("📄 Loading DHL official documents...\n")
for name, path in pdf_files.items():
    if os.path.exists(path):
        loader = PyPDFLoader(path)
        docs = loader.load()
        
        # Add source metadata to each document
        for doc in docs:
            doc.metadata["source_name"] = name
        
        all_documents.extend(docs)
        print(f"✅ {name}: {len(docs)} pages")
    else:
        print(f"❌ {name}: File not found")

print(f"\n📊 Total loaded: {len(all_documents)} pages")

In [ ]:
# Preview loaded document content
print("=" * 60)
print("Document Content Preview (First 500 characters of page 1)")
print("=" * 60)
print(all_documents[0].page_content[:500])
print("\n...\n")
print(f"Metadata: {all_documents[0].metadata}")

---
### Step 4: Semantic Chunking

**Why Semantic Chunking instead of fixed-size chunking?**

| Method | How it works | Pros | Cons |
|--------|-------------|------|------|
| **Fixed-size** | Split every N characters | Fast, simple | Cuts sentences, ignores meaning |
| **Semantic** | Split when meaning changes | Preserves context, smarter boundaries | Slower (needs embeddings) |

**How Semantic Chunking works:**
```
Sentence 1: "DHL Express delivers worldwide."     ─┐
Sentence 2: "We offer next-day delivery."         ─┤ Similar → Same chunk
Sentence 3: "Tracking is available online."       ─┘
                                                    ← Semantic break detected
Sentence 4: "Prohibited items include weapons."   ─┐
Sentence 5: "Explosives are not allowed."         ─┤ Similar → New chunk
```

In [ ]:
# Step 4: SPLIT - Semantic Chunking
from langchain_experimental.text_splitter import SemanticChunker
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings

# Initialize embeddings for semantic chunking
OLLAMA_BASE_URL = "http://localhost:11434"
chunking_embeddings = OllamaEmbeddings(
    model="mistral",
    base_url=OLLAMA_BASE_URL
)

# Method 1: Semantic Chunker (recommended)
# Splits based on semantic similarity between sentences
semantic_splitter = SemanticChunker(
    embeddings=chunking_embeddings,
    breakpoint_threshold_type="percentile",  # Options: "percentile", "standard_deviation", "interquartile"
    breakpoint_threshold_amount=85,          # Higher = fewer, larger chunks
)

print("🧠 Using Semantic Chunking...")
print("   Breakpoint type: percentile")
print("   Threshold: 85 (meaning: split when similarity drops below 85th percentile)")

# Combine all document text for semantic chunking
# SemanticChunker works on text, so we need to process and preserve metadata
chunks = []
for doc in all_documents:
    try:
        # Split this document semantically
        doc_chunks = semantic_splitter.create_documents(
            texts=[doc.page_content],
            metadatas=[doc.metadata]
        )
        chunks.extend(doc_chunks)
    except Exception as e:
        # Fallback to simple splitting if semantic fails for this doc
        print(f"   ⚠️ Fallback for {doc.metadata.get('source_name', 'unknown')}: {e}")
        fallback_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
        doc_chunks = fallback_splitter.split_documents([doc])
        chunks.extend(doc_chunks)

print(f"\n📊 Semantic Chunking Results:")
print(f"   Original documents: {len(all_documents)} pages")
print(f"   After chunking: {len(chunks)} chunks")
print(f"   Average chunks per page: {len(chunks)/len(all_documents):.1f}")

In [ ]:
# Preview semantic chunking results - compare chunk sizes
print("=" * 60)
print("Semantic Chunk Analysis")
print("=" * 60)

# Analyze chunk size distribution
chunk_sizes = [len(chunk.page_content) for chunk in chunks]
print(f"\n📏 Chunk Size Statistics:")
print(f"   Min: {min(chunk_sizes)} chars")
print(f"   Max: {max(chunk_sizes)} chars")
print(f"   Average: {sum(chunk_sizes)/len(chunk_sizes):.0f} chars")
print(f"   Median: {sorted(chunk_sizes)[len(chunk_sizes)//2]} chars")

print("\n📄 Sample Chunks (showing semantic boundaries):")
print("-" * 60)

for i, chunk in enumerate(chunks[:3]):
    print(f"\n--- Chunk {i+1} ({len(chunk.page_content)} chars) ---")
    print(f"Source: {chunk.metadata.get('source_name', 'unknown')}")
    # Show first and last 150 chars to see boundaries
    content = chunk.page_content
    if len(content) > 350:
        print(f"Start: {content[:150]}...")
        print(f"...End: ...{content[-150:]}")
    else:
        print(f"Content: {content}")

---
### Step 5: Generate Embeddings

Convert text into numerical vectors. Semantically similar text will have similar vectors.

```
"DHL delivers in 24 hours" → [0.12, -0.34, 0.56, ..., 0.08] (768 dimensions)
"Express shipping next day" → [0.11, -0.32, 0.55, ..., 0.09]  ← Similar meaning = Similar vectors
```

In [ ]:
# Step 5: EMBED - Generate Vector Embeddings
from langchain_ollama import OllamaEmbeddings

# Initialize embeddings using local Ollama model
embeddings = OllamaEmbeddings(
    model="mistral",
    base_url=OLLAMA_BASE_URL
)

# Test embedding generation
test_text = "What is DHL delivery time?"
test_embedding = embeddings.embed_query(test_text)

print(f"✅ Embedding model initialized successfully")
print(f"\nTest text: '{test_text}'")
print(f"Vector dimensions: {len(test_embedding)}")
print(f"First 10 values: {test_embedding[:10]}")

---
### Step 6: Store in Vector Database

ChromaDB storage structure:
```
┌──────┬─────────────────────────┬──────────────────────┐
│  ID  │  Vector (768 dims)      │  Document            │
├──────┼─────────────────────────┼──────────────────────┤
│  1   │  [0.12, -0.34, ...]     │  "DHL Express..."    │
│  2   │  [0.08, -0.21, ...]     │  "Customs guide..."  │
└──────┴─────────────────────────┴──────────────────────┘
```

In [ ]:
# Step 6: STORE - Create Vector Database
from langchain_chroma import Chroma

# Persistent storage directory
PERSIST_DIRECTORY = "./chroma_db"

print("🚀 Creating vector database (this may take a few minutes)...")
print(f"   Processing {len(chunks)} chunks...")

# Create Chroma vector store
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="dhl_knowledge",
    persist_directory=PERSIST_DIRECTORY
)

print(f"\n✅ Vector database created successfully!")
print(f"   Storage location: {PERSIST_DIRECTORY}")
print(f"   Document count: {len(chunks)} chunks")

---
## Phase 2: Online Query Pipeline
### Retrieve → Generate

---
### Step 7: Create Retriever

User query → Vectorize → Find most similar documents in database

```
User: "What are prohibited items?"
        ↓ Vectorize
    [0.14, -0.33, 0.54, ...]
        ↓ Cosine similarity search
    Return Top-K most relevant chunks
```

In [ ]:
# Step 7: RETRIEVE - Create Retriever

# Create retriever that returns top 10 most similar documents
# More documents = higher recall, but may include less relevant content
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 10}  # Increased from 5 to 10 for better keyword coverage
)

print("✅ Retriever created successfully")
print("   Search type: similarity (cosine)")
print("   Results returned: Top 10")

In [ ]:
# Test retrieval performance
test_queries = [
    "What items are prohibited for shipping?",
    "How do I file a claim for lost package?",
    "What are the customs clearance procedures?"
]

print("=" * 60)
print("Retrieval Test")
print("=" * 60)

for query in test_queries:
    print(f"\n❓ Query: {query}")
    docs = retriever.invoke(query)
    print(f"📄 Retrieved {len(docs)} relevant documents:")
    for i, doc in enumerate(docs[:2]):  # Show only top 2
        print(f"   [{i+1}] {doc.page_content[:150]}...")

---
### Step 8: Build LangGraph Agent

LangGraph ReAct Agent = Reasoning + Acting

```
User Question
    ↓
Agent Thinks: Which tool should I use?
    ↓
Execute Tool (search knowledge / check status / estimate cost)
    ↓
LLM generates answer based on tool results
```

In [ ]:
# Step 8: Define Agent Tools
from langchain_core.tools import tool
import pandas as pd
from datetime import datetime, timedelta

# Tool 1: Search DHL Knowledge Base
@tool
def search_dhl_knowledge(query: str) -> str:
    """Search DHL official documentation for information about shipping, customs, terms, etc."""
    docs = retriever.invoke(query)
    if not docs:
        return "No relevant information found."
    
    context = "\n\n".join([
        f"[Source: {doc.metadata.get('source_name', 'unknown')}]\n{doc.page_content}" 
        for doc in docs[:3]
    ])
    return f"DHL Documentation:\n{context}"

# Tool 2: Check Shipment Status (Simulated Data)
shipments_data = pd.DataFrame({
    "tracking_id": ["DHL001", "DHL002", "DHL003", "DHL004", "DHL005"],
    "origin": ["New York", "Los Angeles", "Chicago", "Houston", "Miami"],
    "destination": ["London", "Tokyo", "Berlin", "Paris", "Sydney"],
    "status": ["IN_TRANSIT", "OUT_FOR_DELIVERY", "DELIVERED", "CUSTOMS_HOLD", "EXCEPTION"],
    "weight_kg": [2.5, 5.0, 1.2, 8.3, 3.7],
    "ship_date": [datetime.now() - timedelta(days=i) for i in range(2, 7)],
    "eta": [datetime.now() + timedelta(days=i) for i in range(1, 6)],
})

@tool
def check_shipment(tracking_id: str) -> str:
    """Check the status of a DHL shipment by tracking ID."""
    shipment = shipments_data[shipments_data['tracking_id'] == tracking_id.upper()]
    
    if shipment.empty:
        return f"Shipment {tracking_id} not found."
    
    s = shipment.iloc[0]
    return f"""
📦 Shipment: {s['tracking_id']}
Route: {s['origin']} → {s['destination']}
Status: {s['status']}
Weight: {s['weight_kg']} kg
Shipped: {s['ship_date'].strftime('%Y-%m-%d')}
ETA: {s['eta'].strftime('%Y-%m-%d')}
""".strip()

# Tool 3: Estimate Shipping Cost
@tool  
def estimate_shipping_cost(origin: str, destination: str, weight_kg: float) -> str:
    """Estimate DHL shipping cost based on origin, destination and weight."""
    base_rate = 25.0
    weight_rate = 5.0  # per kg
    international_surcharge = 15.0 if origin != destination else 0
    
    total = base_rate + (weight_kg * weight_rate) + international_surcharge
    
    return f"""
💰 Shipping Cost Estimate:
Route: {origin} → {destination}
Weight: {weight_kg} kg
Base Rate: ${base_rate:.2f}
Weight Charge: ${weight_kg * weight_rate:.2f}
International: ${international_surcharge:.2f}
━━━━━━━━━━━━━━━━━━
Total: ${total:.2f}
""".strip()

# Tool list
tools = [search_dhl_knowledge, check_shipment, estimate_shipping_cost]

print(f"✅ Created {len(tools)} tools:")
for t in tools:
    print(f"   - {t.name}: {t.description}")

In [ ]:
# Step 9: Create LangGraph ReAct Agent
from langchain_ollama import ChatOllama
from langgraph.prebuilt import create_react_agent

# Initialize LLM
llm = ChatOllama(
    model="mistral",
    base_url=OLLAMA_BASE_URL,
    temperature=0.3  # Lower temperature = more consistent output
)

# Create ReAct Agent
agent = create_react_agent(llm, tools)

print("✅ LangGraph ReAct Agent created successfully")
print("\n📋 Agent Configuration:")
print(f"   LLM: Mistral (Ollama)")
print(f"   Tools: {len(tools)}")
print(f"   Framework: LangGraph ReAct")

In [ ]:
# Query function wrapper
def ask_dhl(question: str) -> str:
    """
    Ask the DHL RAG Agent a question.
    
    Example questions:
    - "What items cannot be shipped via DHL?"
    - "Check status of DHL001"
    - "How much to ship 5kg from New York to London?"
    """
    print(f"\n{'='*60}")
    print(f"❓ Question: {question}")
    print(f"{'='*60}")
    
    response = agent.invoke({
        "messages": [{"role": "user", "content": question}]
    })
    
    answer = response["messages"][-1].content
    print(f"\n🤖 Answer:\n{answer}")
    return answer

---
## Test the RAG System

In [ ]:
# Test 1: Knowledge Base Search
ask_dhl("What items are prohibited from shipping with DHL?")

In [ ]:
# Test 2: Shipment Status Query
ask_dhl("What is the status of shipment DHL004?")

In [ ]:
# Test 3: Cost Estimation
ask_dhl("How much would it cost to ship a 3kg package from New York to London?")

In [ ]:
# Test 4: Complex Query (requires knowledge base search)
ask_dhl("What should I do if my shipment is held at customs? What documents do I need?")

---
## Phase 3: Evaluation Framework

Two dimensions of RAG evaluation:
1. **Retrieval Evaluation** - Are the retrieved documents relevant?
2. **Generation Evaluation** - Is the generated answer correct?

### Metrics:
| Metric | Description |
|--------|-------------|
| **Recall@K** | Proportion of queries where correct document is in top K results |
| **MRR** | Mean Reciprocal Rank - average of 1/rank of first correct result |
| **Answer Accuracy** | LLM-judged correctness of generated answers |
| **Faithfulness** | Whether answer is grounded in retrieved documents |

In [ ]:
# Step 10: Create Test Dataset (Ground Truth)
# Each test case contains: question, expected keywords, expected source document, category

test_dataset = [
    {
        "id": 1,
        "question": "What items are prohibited from shipping with DHL?",
        "expected_keywords": ["dangerous", "prohibited", "weapons", "explosives", "flammable", "illegal"],
        "expected_source": "dhl_express_terms",
        "category": "prohibited_items"
    },
    {
        "id": 2,
        "question": "What is DHL's liability limit for lost packages?",
        "expected_keywords": ["liability", "limit", "compensation", "claim", "value", "SDR", "dollar"],
        "expected_source": "dhl_express_terms",
        "category": "liability"
    },
    {
        "id": 3,
        "question": "How do I file a claim for a damaged shipment?",
        "expected_keywords": ["claim", "damage", "file", "notify", "days", "written"],
        "expected_source": "dhl_express_terms",
        "category": "claims"
    },
    {
        "id": 4,
        "question": "What documents are required for customs clearance?",
        "expected_keywords": ["customs", "documents", "invoice", "declaration", "clearance", "import"],
        "expected_source": "dhl_customs_guide",
        "category": "customs"
    },
    {
        "id": 5,
        "question": "What is the process for importing goods to the US?",
        "expected_keywords": ["import", "US", "customs", "duty", "tariff", "CBP", "broker"],
        "expected_source": "dhl_customs_guide",
        "category": "customs"
    },
    {
        "id": 6,
        "question": "What are the delivery time guarantees for DHL Express?",
        "expected_keywords": ["delivery", "time", "guarantee", "express", "business days", "service"],
        "expected_source": "dhl_express_terms",
        "category": "delivery"
    },
    {
        "id": 7,
        "question": "How does DHL handle shipments that cannot be delivered?",
        "expected_keywords": ["undeliverable", "return", "storage", "dispose", "address", "contact"],
        "expected_source": "dhl_express_terms",
        "category": "delivery"
    },
    {
        "id": 8,
        "question": "What are the packaging requirements for DHL shipments?",
        "expected_keywords": ["packaging", "pack", "secure", "box", "label", "protect"],
        "expected_source": "dhl_express_terms",
        "category": "packaging"
    },
    {
        "id": 9,
        "question": "What duties and taxes apply to international shipments?",
        "expected_keywords": ["duties", "taxes", "customs", "import", "tariff", "fee"],
        "expected_source": "dhl_customs_guide",
        "category": "customs"
    },
    {
        "id": 10,
        "question": "What is DHL's privacy policy regarding shipment data?",
        "expected_keywords": ["privacy", "data", "information", "personal", "protect", "confidential"],
        "expected_source": "dhl_express_terms",
        "category": "privacy"
    },
]

print(f"✅ Test dataset created: {len(test_dataset)} test cases")
print("\n📋 Category distribution:")
categories = {}
for item in test_dataset:
    cat = item["category"]
    categories[cat] = categories.get(cat, 0) + 1
for cat, count in categories.items():
    print(f"   - {cat}: {count} questions")

In [ ]:
# Step 11: Retrieval Evaluation (Improved Keyword Matching)

def flexible_keyword_match(keyword: str, content: str) -> bool:
    """
    Flexible keyword matching:
    1. Exact match
    2. Partial match (keyword is substring)
    3. Stem match (e.g., "weapon" matches "weapons")
    """
    keyword = keyword.lower()
    content = content.lower()
    
    # 1. Exact or substring match
    if keyword in content:
        return True
    
    # 2. Stem matching (simple: check if keyword root appears)
    # Handle common suffixes: s, es, ed, ing, tion, ly
    stems = [keyword]
    if keyword.endswith('s'):
        stems.append(keyword[:-1])  # weapons -> weapon
    if keyword.endswith('es'):
        stems.append(keyword[:-2])  # boxes -> box
    if keyword.endswith('ed'):
        stems.append(keyword[:-2])  # damaged -> damag
    if keyword.endswith('ing'):
        stems.append(keyword[:-3])  # shipping -> shipp
    
    # Also try adding common suffixes
    stems.extend([keyword + 's', keyword + 'es', keyword + 'ed', keyword + 'ing'])
    
    for stem in stems:
        if stem in content:
            return True
    
    return False


def evaluate_retrieval(test_data, retriever, k=5):
    """
    Evaluate retrieval quality with improved keyword matching.
    
    Metrics:
    - Recall@K: Proportion of queries with correct source in top K
    - MRR: Mean Reciprocal Rank
    - Keyword Recall: Proportion of expected keywords found (flexible matching)
    """
    results = []
    
    for item in test_data:
        question = item["question"]
        expected_source = item["expected_source"]
        expected_keywords = item["expected_keywords"]
        
        # Retrieve documents
        retrieved_docs = retriever.invoke(question)
        
        # Check source match
        retrieved_sources = [doc.metadata.get("source_name", "") for doc in retrieved_docs]
        source_hit = expected_source in retrieved_sources
        
        # Calculate source rank (for MRR)
        source_rank = None
        for i, source in enumerate(retrieved_sources):
            if source == expected_source:
                source_rank = i + 1
                break
        
        # Improved keyword matching (flexible)
        all_content = " ".join([doc.page_content for doc in retrieved_docs])
        keyword_hits = sum(1 for kw in expected_keywords if flexible_keyword_match(kw, all_content))
        keyword_recall = keyword_hits / len(expected_keywords)
        
        # Track which keywords matched/missed for debugging
        matched_kw = [kw for kw in expected_keywords if flexible_keyword_match(kw, all_content)]
        missed_kw = [kw for kw in expected_keywords if not flexible_keyword_match(kw, all_content)]
        
        results.append({
            "id": item["id"],
            "question": question[:50] + "...",
            "source_hit": source_hit,
            "source_rank": source_rank,
            "keyword_recall": keyword_recall,
            "matched_keywords": matched_kw,
            "missed_keywords": missed_kw,
            "category": item["category"]
        })
    
    # Calculate overall metrics
    recall_at_k = sum(1 for r in results if r["source_hit"]) / len(results)
    mrr = sum(1/r["source_rank"] for r in results if r["source_rank"]) / len(results)
    avg_keyword_recall = sum(r["keyword_recall"] for r in results) / len(results)
    
    return {
        "results": results,
        "metrics": {
            "Recall@K": recall_at_k,
            "MRR": mrr,
            "Keyword_Recall": avg_keyword_recall
        }
    }

# Run retrieval evaluation
print("🔍 Running retrieval evaluation (with flexible keyword matching)...")
print("=" * 60)

retrieval_eval = evaluate_retrieval(test_dataset, retriever, k=5)

print("\n📊 Retrieval Evaluation Results:")
print(f"   Recall@5: {retrieval_eval['metrics']['Recall@K']:.1%}")
print(f"   MRR: {retrieval_eval['metrics']['MRR']:.3f}")
print(f"   Keyword Recall: {retrieval_eval['metrics']['Keyword_Recall']:.1%}")

print("\n📋 Detailed Results:")
for r in retrieval_eval["results"]:
    status = "✅" if r["source_hit"] else "❌"
    print(f"\n   {status} Q{r['id']}: {r['question']}")
    print(f"      Keywords: {r['keyword_recall']:.0%} | ✓ {r['matched_keywords'][:3]}")
    if r["missed_keywords"]:
        print(f"      Missed: {r['missed_keywords'][:3]}")

In [ ]:
# Step 12: Generation Evaluation (LLM-as-Judge)

def evaluate_answer_with_llm(question: str, answer: str, expected_keywords: list) -> dict:
    """
    Use LLM to evaluate answer quality.
    
    Scoring dimensions:
    1. Relevance (1-5): Is the answer relevant to the question?
    2. Completeness (1-5): Does the answer fully address the question?
    3. Accuracy (1-5): Does the answer include expected concepts?
    """
    eval_prompt = f"""You are an evaluation assistant. Rate the following answer on a scale of 1-5 for each criterion.

Question: {question}

Answer: {answer}

Expected to mention these concepts: {', '.join(expected_keywords)}

Rate the answer (respond with ONLY numbers separated by commas, e.g., "4,3,5"):
1. Relevance (1-5): Is the answer relevant to the question?
2. Completeness (1-5): Does the answer fully address the question?
3. Accuracy (1-5): Does the answer include the expected concepts?

Scores:"""

    try:
        response = llm.invoke(eval_prompt)
        scores_text = response.content.strip()
        
        # Parse scores
        import re
        numbers = re.findall(r'\d+', scores_text)
        if len(numbers) >= 3:
            relevance = min(int(numbers[0]), 5)
            completeness = min(int(numbers[1]), 5)
            accuracy = min(int(numbers[2]), 5)
        else:
            relevance = completeness = accuracy = 3  # Default scores
            
        return {
            "relevance": relevance,
            "completeness": completeness,
            "accuracy": accuracy,
            "average": (relevance + completeness + accuracy) / 3
        }
    except Exception as e:
        return {"relevance": 0, "completeness": 0, "accuracy": 0, "average": 0, "error": str(e)}


def run_full_evaluation(test_data, agent, retriever, num_samples=5):
    """
    Run complete RAG evaluation (retrieval + generation).
    """
    results = []
    
    print(f"🧪 Evaluating {num_samples} test cases...")
    print("=" * 60)
    
    for i, item in enumerate(test_data[:num_samples]):
        print(f"\n[{i+1}/{num_samples}] Evaluating: {item['question'][:50]}...")
        
        # 1. Get Agent answer
        try:
            response = agent.invoke({
                "messages": [{"role": "user", "content": item["question"]}]
            })
            answer = response["messages"][-1].content
        except Exception as e:
            answer = f"Error: {e}"
        
        # 2. Retrieval evaluation
        retrieved_docs = retriever.invoke(item["question"])
        retrieved_sources = [doc.metadata.get("source_name", "") for doc in retrieved_docs]
        source_hit = item["expected_source"] in retrieved_sources
        
        # 3. Keyword matching (simple method)
        keyword_hits = sum(1 for kw in item["expected_keywords"] if kw.lower() in answer.lower())
        keyword_score = keyword_hits / len(item["expected_keywords"])
        
        # 4. LLM evaluation
        llm_scores = evaluate_answer_with_llm(item["question"], answer, item["expected_keywords"])
        
        results.append({
            "id": item["id"],
            "question": item["question"],
            "answer": answer[:200] + "..." if len(answer) > 200 else answer,
            "source_hit": source_hit,
            "keyword_score": keyword_score,
            "llm_scores": llm_scores,
            "category": item["category"]
        })
        
        print(f"   Source Hit: {'✅' if source_hit else '❌'} | Keywords: {keyword_score:.0%} | LLM Avg: {llm_scores['average']:.1f}/5")
    
    return results


# Run full evaluation (using first 5 test cases to save time)
print("\n🚀 Starting Full RAG Evaluation")
eval_results = run_full_evaluation(test_dataset, agent, retriever, num_samples=5)

In [ ]:
# Step 13: Generate Evaluation Report

def generate_evaluation_report(eval_results, retrieval_metrics):
    """Generate comprehensive evaluation report."""
    
    print("\n" + "=" * 60)
    print("📊 RAG SYSTEM EVALUATION REPORT")
    print("=" * 60)
    
    # 1. Retrieval Metrics
    print("\n【1. Retrieval Evaluation】")
    print(f"   Recall@5:        {retrieval_metrics['Recall@K']:.1%}")
    print(f"   MRR:             {retrieval_metrics['MRR']:.3f}")
    print(f"   Keyword Recall:  {retrieval_metrics['Keyword_Recall']:.1%}")
    
    # 2. Generation Metrics
    print("\n【2. Generation Evaluation】")
    
    avg_keyword = sum(r["keyword_score"] for r in eval_results) / len(eval_results)
    avg_relevance = sum(r["llm_scores"]["relevance"] for r in eval_results) / len(eval_results)
    avg_completeness = sum(r["llm_scores"]["completeness"] for r in eval_results) / len(eval_results)
    avg_accuracy = sum(r["llm_scores"]["accuracy"] for r in eval_results) / len(eval_results)
    avg_overall = sum(r["llm_scores"]["average"] for r in eval_results) / len(eval_results)
    
    print(f"   Keyword Match:   {avg_keyword:.1%}")
    print(f"   Relevance:       {avg_relevance:.1f}/5")
    print(f"   Completeness:    {avg_completeness:.1f}/5")
    print(f"   Accuracy:        {avg_accuracy:.1f}/5")
    print(f"   Overall Score:   {avg_overall:.1f}/5 ({avg_overall/5*100:.0f}%)")
    
    # 3. Category Analysis
    print("\n【3. Category Analysis】")
    categories = {}
    for r in eval_results:
        cat = r["category"]
        if cat not in categories:
            categories[cat] = []
        categories[cat].append(r["llm_scores"]["average"])
    
    for cat, scores in categories.items():
        avg = sum(scores) / len(scores)
        print(f"   {cat}: {avg:.1f}/5")
    
    # 4. Summary
    print("\n【4. Summary】")
    print("=" * 60)
    
    overall_score = (retrieval_metrics['Recall@K'] + avg_overall/5) / 2 * 100
    
    if overall_score >= 80:
        grade = "A (Excellent)"
    elif overall_score >= 70:
        grade = "B (Good)"
    elif overall_score >= 60:
        grade = "C (Satisfactory)"
    else:
        grade = "D (Needs Improvement)"
    
    print(f"   Overall Score: {overall_score:.0f}/100 - {grade}")
    print(f"   Test Samples: {len(eval_results)}")
    print(f"   Knowledge Base: DHL Official Docs (Express, Customs, eCommerce)")
    
    return {
        "retrieval": retrieval_metrics,
        "generation": {
            "keyword_match": avg_keyword,
            "relevance": avg_relevance,
            "completeness": avg_completeness,
            "accuracy": avg_accuracy,
            "overall": avg_overall
        },
        "overall_score": overall_score,
        "grade": grade
    }

# Generate report
final_report = generate_evaluation_report(eval_results, retrieval_eval["metrics"])

In [ ]:
# Step 14: Visualize Evaluation Results

import matplotlib.pyplot as plt

def plot_evaluation_results(eval_results, retrieval_metrics):
    """Visualize evaluation results."""
    
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    
    # Chart 1: Retrieval Metrics
    ax1 = axes[0]
    metrics = ["Recall@5", "MRR", "Keyword\nRecall"]
    values = [
        retrieval_metrics["Recall@K"], 
        retrieval_metrics["MRR"], 
        retrieval_metrics["Keyword_Recall"]
    ]
    colors = ["#2ecc71" if v >= 0.7 else "#f39c12" if v >= 0.5 else "#e74c3c" for v in values]
    bars = ax1.bar(metrics, values, color=colors)
    ax1.set_ylim(0, 1)
    ax1.set_title("Retrieval Metrics", fontsize=12, fontweight="bold")
    ax1.set_ylabel("Score")
    for bar, v in zip(bars, values):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
                f"{v:.0%}", ha="center", fontsize=10)
    
    # Chart 2: Generation Quality Scores
    ax2 = axes[1]
    gen_metrics = ["Relevance", "Complete-\nness", "Accuracy"]
    gen_values = [
        sum(r["llm_scores"]["relevance"] for r in eval_results) / len(eval_results),
        sum(r["llm_scores"]["completeness"] for r in eval_results) / len(eval_results),
        sum(r["llm_scores"]["accuracy"] for r in eval_results) / len(eval_results),
    ]
    colors = ["#3498db"] * 3
    bars = ax2.bar(gen_metrics, gen_values, color=colors)
    ax2.set_ylim(0, 5)
    ax2.set_title("Generation Quality (LLM Judge)", fontsize=12, fontweight="bold")
    ax2.set_ylabel("Score (1-5)")
    for bar, v in zip(bars, gen_values):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, 
                f"{v:.1f}", ha="center", fontsize=10)
    
    # Chart 3: Score by Question
    ax3 = axes[2]
    questions = [f"Q{r['id']}" for r in eval_results]
    scores = [r["llm_scores"]["average"] for r in eval_results]
    colors = ["#2ecc71" if s >= 4 else "#f39c12" if s >= 3 else "#e74c3c" for s in scores]
    bars = ax3.bar(questions, scores, color=colors)
    ax3.set_ylim(0, 5)
    ax3.set_title("Score by Question", fontsize=12, fontweight="bold")
    ax3.set_ylabel("Average Score (1-5)")
    ax3.axhline(y=3.5, color="gray", linestyle="--", alpha=0.5, label="Passing threshold")
    
    plt.tight_layout()
    plt.savefig("evaluation_results.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("\n📈 Chart saved: evaluation_results.png")

# Generate visualization
try:
    plot_evaluation_results(eval_results, retrieval_eval["metrics"])
except Exception as e:
    print(f"⚠️ Visualization failed: {e}")

---
## System Architecture Summary

```
┌─────────────────────────────────────────────────────────────┐
│                  OFFLINE PHASE (Indexing)                   │
├─────────────────────────────────────────────────────────────┤
│  DHL PDFs → Load → Split → Embed → Store (ChromaDB)        │
└─────────────────────────────────────────────────────────────┘
                              ↓
┌─────────────────────────────────────────────────────────────┐
│                  ONLINE PHASE (Querying)                    │
├─────────────────────────────────────────────────────────────┤
│  User Question                                              │
│      ↓                                                      │
│  LangGraph ReAct Agent                                      │
│      ├── Tool 1: search_dhl_knowledge (Vector Retrieval)    │
│      ├── Tool 2: check_shipment (Status Query)              │
│      └── Tool 3: estimate_shipping_cost (Cost Calculator)   │
│      ↓                                                      │
│  Mistral LLM Generates Response                             │
└─────────────────────────────────────────────────────────────┘
                              ↓
┌─────────────────────────────────────────────────────────────┐
│                  EVALUATION PHASE                           │
├─────────────────────────────────────────────────────────────┤
│  Test Dataset (10 cases) → Retrieval Eval → Generation Eval │
│  Metrics: Recall@K, MRR, LLM-as-Judge Scoring               │
└─────────────────────────────────────────────────────────────┘
```